In [1]:
# Install necessary packages
%pip install langchain sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import os
import json
from datetime import datetime
from typing import List, Dict


In [3]:
folder_paths = ["../data/cleaned_companydata", "../data/cleaned_date"]

# Load JSON data
json_data = []
for folder_path in folder_paths:
    json_data.extend(load_json_files(folder_path))

# Print the first record to check the loaded data
if json_data:
    print("First loaded record:", json_data[0])  # Prints the first JSON record
else:
    print("No data found in the specified folders.")


NameError: name 'load_json_files' is not defined

In [ ]:
# Function to Load All JSON Files from the Specified Folders
def load_json_files(folder_paths: str) -> List[Dict]:
    """Loads all JSON files from a folder and combines them into a list."""
    data = []
    for file in os.listdir(folder_paths):
        if file.endswith(".json"):
            with open(os.path.join(folder_paths, file), "r", encoding="utf-8") as f:
                data.extend(json.load(f))
    return data


In [ ]:
# Function to Standardize Date Format
def normalize_date(date_str: str, date_format: str = "%Y-%m-%d") -> str:
    """Converts various date formats to a standard format."""
    try:
        return datetime.strptime(date_str, date_format).strftime("%Y-%m-%d")
    except ValueError:
        return None


In [ ]:
def process_json_data(json_data: List[Dict], date_key: str = "date") -> str:
    cleaned_pages = []
    for record in json_data:
        if isinstance(record, dict):
            if date_key in record:
                record[date_key] = normalize_date(record[date_key])
            cleaned_pages.append(record.get("content", ""))
    return "\n".join(cleaned_pages)


In [33]:
from langchain.schema import Document
# Function to Split Text into Chunks
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document  # Correct import

def chunk_and_embed(text: str, chunk_size: int = 1000, chunk_overlap: int = 200):
    text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separator="."
    )
    chunks = text_splitter.split_text(text)
    return [Document(page_content=chunk) for chunk in chunks]


In [34]:
all_json_data = []
for path in folder_paths:
	json_data = load_json_files(path)
	all_json_data.extend(json_data)

processed_text = process_json_data(all_json_data)
chunks = chunk_and_embed(processed_text)
